# Word Embeddings — Words as Vectors with Meaning

BoW and TF-IDF count words but don't understand them. "happy" and "joyful" are just two
different columns in a sparse matrix — no relationship at all.

**Word embeddings** fix this: every word becomes a dense vector (50–300 dimensions) where
words with similar meanings land near each other in vector space. This was a revolution.

---
## The Big Idea

Instead of sparse one-hot vectors (10,000+ dimensions, mostly zeros), learn **dense** vectors
that capture semantic relationships:

```
king  - man  + woman ≈ queen
paris - france + italy ≈ rome
walked - walking + swimming ≈ swam
```

These aren't hand-coded rules — they **emerge** from training on large text corpora.
The model learns that "king" and "queen" differ in the same way "man" and "woman" do,
because they appear in similar contexts with a gender shift.

**Key insight (Distributional Hypothesis):** "You shall know a word by the company it keeps."
Words that appear in similar contexts have similar meanings.

In [ ]:
import numpy as np

one_hot_cat = np.array([1, 0, 0, 0, 0, 0, 0, 0])  # 8 words in vocab
one_hot_dog = np.array([0, 1, 0, 0, 0, 0, 0, 0])
one_hot_car = np.array([0, 0, 1, 0, 0, 0, 0, 0])

from numpy.linalg import norm

cos_sim = lambda a, b: np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

print("One-hot vectors:")
print(f"  cat·dog similarity: {cos_sim(one_hot_cat, one_hot_dog):.2f}")
print(f"  cat·car similarity: {cos_sim(one_hot_cat, one_hot_car):.2f}")
print("  Every pair is equally unrelated — cat is no more similar to dog than to car.\n")

embed_cat = np.array([0.8, 0.2, -0.1, 0.9])  # hypothetical 4D embedding
embed_dog = np.array([0.7, 0.3, -0.2, 0.8])
embed_car = np.array([-0.5, 0.9, 0.7, -0.3])

print("Embedding vectors:")
print(f"  cat·dog similarity: {cos_sim(embed_cat, embed_dog):.3f}  ← animals are close!")
print(f"  cat·car similarity: {cos_sim(embed_cat, embed_car):.3f}  ← different concepts, far apart")

---
## Word2Vec — The Breakthrough (2013)

Mikolov et al. showed you could train word vectors with a simple neural network on raw text.
Two architectures:

| Model | Task | Intuition |
|---|---|---|
| **Skip-gram** | Given center word, predict context words | "Given 'sat', predict 'cat', 'on', 'mat'" |
| **CBOW** | Given context words, predict center word | "Given 'cat', 'on', 'mat', predict 'sat'" |

```
"The cat sat on the mat"
        ↓ (window=2)
Skip-gram: sat → [cat, on] (predict neighbors from center)
CBOW:      [cat, on] → sat (predict center from neighbors)
```

The hidden layer weights become the word vectors. Skip-gram works better for rare words;
CBOW is faster to train.

In [ ]:
from gensim.models import Word2Vec

sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "sat", "on", "the", "rug"],
    ["the", "cat", "chased", "the", "mouse"],
    ["the", "dog", "chased", "the", "cat"],
    ["a", "cat", "is", "a", "pet"],
    ["a", "dog", "is", "a", "pet"],
    ["the", "king", "ruled", "the", "kingdom"],
    ["the", "queen", "ruled", "the", "kingdom"],
    ["the", "prince", "became", "the", "king"],
    ["the", "princess", "became", "the", "queen"],
    ["paris", "is", "the", "capital", "of", "france"],
    ["rome", "is", "the", "capital", "of", "italy"],
    ["berlin", "is", "the", "capital", "of", "germany"],
]

model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1,
    epochs=200,
    seed=42
)

print(f"Vocabulary: {list(model.wv.key_to_index.keys())}")
print(f"Vector for 'cat': {model.wv['cat'][:8]}... (50 dimensions)")

In [ ]:
print("Most similar to 'cat':")
for word, score in model.wv.most_similar('cat', topn=5):
    print(f"  {word}: {score:.3f}")

print("\nMost similar to 'king':")
for word, score in model.wv.most_similar('king', topn=5):
    print(f"  {word}: {score:.3f}")

print("\nNote: with such a tiny corpus, results are noisy. Pre-trained models on billions")
print("of words produce much more meaningful relationships.")

---
## Visualizing Word Embeddings

Embeddings live in 50–300 dimensional space. We can project down to 2D with PCA or t-SNE
to see the clusters and relationships.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

words = list(model.wv.key_to_index.keys())
vectors = np.array([model.wv[w] for w in words])

pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)

plt.figure(figsize=(10, 7))

animals = {'cat', 'dog', 'mouse', 'pet'}
royalty = {'king', 'queen', 'prince', 'princess', 'kingdom'}
places = {'paris', 'rome', 'berlin', 'france', 'italy', 'germany', 'capital'}

for word, (x, y) in zip(words, coords):
    if word in animals:
        color = 'green'
    elif word in royalty:
        color = 'red'
    elif word in places:
        color = 'blue'
    else:
        color = 'gray'
    plt.scatter(x, y, c=color, s=60, alpha=0.7)
    plt.annotate(word, (x, y), fontsize=9, ha='center', va='bottom')

plt.title('Word Embeddings Projected to 2D (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Green=animals, Red=royalty, Blue=places, Gray=other")
print("With a bigger corpus, these clusters become much tighter.")

---
## GloVe — Global Vectors for Word Representation

Word2Vec learns from local context windows. **GloVe** (Pennington et al., 2014) combines:
- **Local context** (like Word2Vec)
- **Global co-occurrence statistics** (how often pairs of words appear together across the entire corpus)

It builds a word-word co-occurrence matrix, then factorizes it to get dense vectors.
The result: embeddings that capture both local patterns and corpus-wide statistics.

Pre-trained GloVe vectors are available in several sizes:
- `glove.6B` — trained on 6 billion tokens from Wikipedia + Gigaword (50d, 100d, 200d, 300d)
- `glove.42B` — 42 billion tokens from Common Crawl (300d)
- `glove.840B` — 840 billion tokens (300d)

---
## Using Pre-trained Embeddings

Training your own embeddings requires massive data. In practice, you almost always
start with **pre-trained** vectors and use them directly or fine-tune them.

Gensim provides easy access to pre-trained models.

In [ ]:
import gensim.downloader as api

print("Available pre-trained models:")
models = ['word2vec-google-news-300', 'glove-wiki-gigaword-50', 
          'glove-wiki-gigaword-100', 'glove-wiki-gigaword-300',
          'glove-twitter-25', 'glove-twitter-50']
for m in models:
    print(f"  • {m}")

print("\nLoading glove-wiki-gigaword-50 (~65MB download, smallest useful model)...")
print("This may take a minute on first run.")
glove = api.load('glove-wiki-gigaword-50')
print(f"Loaded! Vocabulary: {len(glove)} words, {glove.vector_size} dimensions")

In [ ]:
print("Most similar to 'king':")
for word, score in glove.most_similar('king', topn=8):
    print(f"  {word}: {score:.3f}")

print("\nMost similar to 'python':")
for word, score in glove.most_similar('python', topn=5):
    print(f"  {word}: {score:.3f}")

In [ ]:
print("Word Analogies (the classic test):\n")

analogies = [
    ('king', 'man', 'woman'),       # king - man + woman = ?
    ('paris', 'france', 'italy'),    # paris - france + italy = ?
    ('good', 'better', 'bad'),       # good:better :: bad:?
    ('japan', 'sushi', 'italy'),     # japan:sushi :: italy:?
]

for pos1, neg, pos2 in analogies:
    result = glove.most_similar(positive=[pos1, pos2], negative=[neg], topn=1)
    word, score = result[0]
    print(f"  {pos1} - {neg} + {pos2} = {word} ({score:.3f})")

In [ ]:
print("Which word doesn't belong?\n")
groups = [
    ['cat', 'dog', 'fish', 'computer'],
    ['paris', 'london', 'berlin', 'banana'],
    ['happy', 'sad', 'angry', 'table'],
]

for group in groups:
    odd = glove.doesnt_match(group)
    print(f"  {group} → odd one out: '{odd}'")

---
## Visualizing Pre-trained Embeddings

In [ ]:
from sklearn.manifold import TSNE

word_groups = {
    'animals': ['cat', 'dog', 'fish', 'bird', 'horse', 'lion', 'tiger'],
    'countries': ['france', 'germany', 'italy', 'spain', 'japan', 'china', 'india'],
    'emotions': ['happy', 'sad', 'angry', 'fear', 'love', 'hate', 'joy'],
    'tech': ['computer', 'software', 'internet', 'algorithm', 'data', 'code', 'program'],
}

all_words = []
all_colors = []
color_map = {'animals': 'green', 'countries': 'blue', 'emotions': 'red', 'tech': 'purple'}

for category, words in word_groups.items():
    for w in words:
        all_words.append(w)
        all_colors.append(color_map[category])

vecs = np.array([glove[w] for w in all_words])

tsne = TSNE(n_components=2, random_state=42, perplexity=8)
coords = tsne.fit_transform(vecs)

plt.figure(figsize=(12, 8))
for i, word in enumerate(all_words):
    plt.scatter(coords[i, 0], coords[i, 1], c=all_colors[i], s=80, alpha=0.7)
    plt.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=10, ha='center', va='bottom')

import matplotlib.patches as mpatches
legend = [mpatches.Patch(color=c, label=cat) for cat, c in color_map.items()]
plt.legend(handles=legend, fontsize=11)
plt.title('GloVe Embeddings Visualized with t-SNE', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Embedding Layers in PyTorch

In neural networks, an **embedding layer** is just a lookup table that maps integer IDs
to dense vectors. These vectors are learned during training.

```
word_id = 42  →  embedding_layer  →  [0.12, -0.34, 0.56, ...] (dense vector)
```

Under the hood: it's a matrix of shape `(vocab_size, embedding_dim)`. Looking up a word
is just indexing into this matrix.

In [ ]:
import torch
import torch.nn as nn

vocab_size = 1000
embed_dim = 64

embedding = nn.Embedding(vocab_size, embed_dim)

print(f"Embedding matrix shape: {embedding.weight.shape}")
print(f"This IS the lookup table: {vocab_size} words × {embed_dim} dimensions\n")

word_ids = torch.tensor([42, 7, 256])
vectors = embedding(word_ids)

print(f"Input: 3 word IDs → {word_ids.tolist()}")
print(f"Output: 3 vectors → shape {vectors.shape}")
print(f"\nVector for word 42: {vectors[0][:8].detach().numpy().round(3)}...")

In [ ]:
class SimpleTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)        # (batch, seq_len) → (batch, seq_len, embed_dim)
        pooled = embedded.mean(dim=1)        # average all word vectors → (batch, embed_dim)
        return self.fc(pooled)               # → (batch, num_classes)

model = SimpleTextClassifier(vocab_size=5000, embed_dim=64, num_classes=2)

fake_input = torch.randint(0, 5000, (4, 10))  # batch of 4 sentences, 10 words each
output = model(fake_input)
print(f"Input shape:  {fake_input.shape}  (4 sentences × 10 words)")
print(f"Output shape: {output.shape}  (4 sentences × 2 classes)")
print(f"\nThe embedding layer's weights are learned during training.")
print(f"Alternatively, you can initialize with pre-trained GloVe/Word2Vec vectors.")

---
## Loading Pre-trained Vectors into PyTorch

In [ ]:
sample_vocab = ['<pad>', '<unk>', 'the', 'cat', 'dog', 'sat', 'on', 'mat']
word2idx = {w: i for i, w in enumerate(sample_vocab)}

pretrained_matrix = np.zeros((len(sample_vocab), 50))
found = 0
for word, idx in word2idx.items():
    if word in glove:
        pretrained_matrix[idx] = glove[word]
        found += 1

print(f"Found {found}/{len(sample_vocab)} words in GloVe")

embedding_layer = nn.Embedding.from_pretrained(
    torch.FloatTensor(pretrained_matrix),
    freeze=False  # False = fine-tune during training; True = keep fixed
)

cat_vec = embedding_layer(torch.tensor([word2idx['cat']]))
dog_vec = embedding_layer(torch.tensor([word2idx['dog']]))

sim = torch.cosine_similarity(cat_vec, dog_vec)
print(f"\ncat·dog similarity (GloVe): {sim.item():.3f}")
print("Pre-trained vectors give meaningful similarities from the start.")

---
## Limitations of Static Embeddings

Word2Vec and GloVe give each word **one** vector, regardless of context.

But words change meaning based on context:

| Sentence | Meaning of "bank" |
|---|---|
| "I deposited money at the **bank**" | Financial institution |
| "We sat on the river **bank**" | Edge of a river |

Both get the SAME vector! This is fundamentally broken.

More examples:
- "The **bat** flew out of the cave" vs "He swung the **bat** at the ball"
- "She **left** the room" vs "Turn **left** at the corner"
- "That's pretty **cool**" vs "The water was **cool**"

In [ ]:
bank_vec = glove['bank']

financial_words = ['money', 'deposit', 'loan', 'finance', 'account']
nature_words = ['river', 'stream', 'water', 'shore', 'creek']

print("'bank' similarity to financial words:")
for w in financial_words:
    sim = np.dot(bank_vec, glove[w]) / (norm(bank_vec) * norm(glove[w]))
    print(f"  bank·{w}: {sim:.3f}")

print("\n'bank' similarity to nature words:")
for w in nature_words:
    sim = np.dot(bank_vec, glove[w]) / (norm(bank_vec) * norm(glove[w]))
    print(f"  bank·{w}: {sim:.3f}")

print("\nThe single vector for 'bank' mixes both meanings together.")
print("We need CONTEXT-DEPENDENT embeddings → Transformers solve this.")

---
## Measuring Embedding Quality

In [ ]:
def document_vector(text, embeddings):
    words = text.lower().split()
    vecs = [embeddings[w] for w in words if w in embeddings]
    if not vecs:
        return np.zeros(embeddings.vector_size)
    return np.mean(vecs, axis=0)

docs = [
    "The cat sat on the mat",
    "A kitten rested on the rug",
    "Stock prices rose sharply today",
    "The financial markets surged upward",
]

doc_vecs = [document_vector(d, glove) for d in docs]

from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(doc_vecs)

import pandas as pd
labels = [d[:25] + '...' for d in docs]
df = pd.DataFrame(sim_matrix.round(3), index=labels, columns=labels)
print("Document similarity (averaged GloVe vectors):\n")
print(df)
print("\nSimilar topics cluster together — even with simple averaging!")

---

## Summary

| Method | Dimension | Captures Meaning | Context-Aware |
|---|---|---|---|
| One-hot | vocab_size (sparse) | ✗ | ✗ |
| BoW/TF-IDF | vocab_size (sparse) | Partially | ✗ |
| Word2Vec | 50-300 (dense) | ✓ | ✗ |
| GloVe | 50-300 (dense) | ✓ | ✗ |

**Key takeaways:**
- Word embeddings represent words as dense vectors where semantics are captured by geometry
- Word2Vec learns from local context; GloVe adds global statistics
- Pre-trained embeddings are powerful — use them as initialization
- The fatal flaw: one vector per word, regardless of context

**Next notebook:** Transformers give every word a DIFFERENT vector depending on context.
This is the architecture that powers GPT, BERT, and every modern language model.